## Instalação e Verificação de Dependências para Especialização

In [1]:
import os
import subprocess

if not os.path.exists("agrofacil-genai"):
    print("A clonar o repositório do projeto...")
    subprocess.run(["git", "clone", "https://github.com/palomari/agrofacil-genai.git"])

if not os.path.exists("base") and os.path.exists("agrofacil-genai"):
    os.chdir("agrofacil-genai")
    print("Diretório alterado para a raiz do projeto: agrofacil-genai")

if os.path.exists("base/documentos_tecnicos.txt"):
    print("Ambiente configurado com sucesso! A pasta 'base' está pronta a ser lida.")
else:
    print("Aviso: Certifique-se de que a estrutura de pastas do repositório está correta.")

Diretório alterado para a raiz do projeto: agrofacil-genai
Ambiente configurado com sucesso! A pasta 'base' está pronta a ser lida.


In [2]:
!pip install -q transformers datasets accelerate peft bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"GPU Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

GPU Disponível: True
Dispositivo: Tesla T4


In [3]:
!pip install -q trl peft bitsandbytes accelerate datasets

## Configuração do Modelo Base e Carregamento de Adaptadores (Fine-tuning)

In [4]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

modelo_base_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    modelo_base_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

dados_treino = {
    "text": [
        "Como o AgroFácil otimiza a irrigação? O AgroFácil utiliza sensores de umidade do solo para acionar as bombas de água somente quando necessário, reduzindo o desperdício hídrico.",
        "Qual a recomendação de adubação? A recomendação para correção de pH do solo ácido deve ser feita utilizando calcário dolomítico com base na análise prévia de solo."
    ]
}
dataset = Dataset.from_dict(dados_treino)

training_args = TrainingArguments(
    output_dir="./resultados_agrofacil",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=3,
    max_steps=10,
    optim="paged_adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Iniciando o Ajuste Fino do Adaptador para o AgroFácil com Qwen2.5...")
trainer.train()

trainer.model.save_pretrained("./agrofacil_adapter")
print("Ajuste Fino concluído e adaptador salvo com sucesso na pasta 'agrofacil_adapter'!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Iniciando o Ajuste Fino do Adaptador para o AgroFácil com Qwen2.5...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,2.822811
2,2.255055
3,1.821959
4,1.475695
5,1.202297
6,0.994814
7,0.835380
8,0.721401
9,0.629138
10,0.573851


Ajuste Fino concluído e adaptador salvo com sucesso na pasta 'agrofacil_adapter'!


## Teste de Resposta Especializada para o Domínio AgroFácil

In [5]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


config_path = "./agrofacil_adapter/adapter_config.json"
with open(config_path, "r") as f:
    config_data = json.load(f)

config_data["task_type"] = "CAUSAL_LM"
with open(config_path, "w") as f:
    json.dump(config_data, f, indent=4)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

modelo_base_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_base_id, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    modelo_base_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


model = PeftModel.from_pretrained(model, "./agrofacil_adapter")
model.eval()


pergunta = "Como o AgroFácil otimiza a irrigação?"
inputs = tokenizer(pergunta, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=64, 
        temperature=0.7, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- Resposta do AgroFácil Especializado ---")
print(resposta)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

--- Resposta do AgroFácil Especializado ---
Como o AgroFácil otimiza a irrigação? O AgroFácil utiliza sensores de umidade do solo para acionar os sprinklers somente quando necessário, reduzindo o desperdício hídrico. Como resultado, ele minimiza as precipitações e reduz as necessidades de água para cultivos agrícolas. Além


## Conclusão da Especialização (Fine-Tuning com LoRA)
- **Quantização:** O modelo base `Qwen2.5-1.5B-Instruct` foi carregado em 4-bit (NF4/BitsAndBytes) para otimização do uso de VRAM no ambiente Google Colab (GPU Tesla T4).
- **Fine-Tuning LoRA:** Aplicado com sucesso nos módulos de atenção e projeção (`q_proj`, `k_proj`, `v_proj`, etc.), reduzindo drasticamente o consumo de memória durante o ajuste fino.
- **Artefatos:** O adaptador treinado foi exportado corretamente com `task_type = "CAUSAL_LM"` no `adapter_config.json`, pronto para ser acoplado no pipeline de inferência.